# LLMs for Business Valuation

## Valuation Fundamentals: DCF Foundation & Challenges


**The Core DCF Formula: Discount Cash Flows**

$$\text{Enterprise Value} = \mathbb{E} \Big(\sum_{t=1}^{T}\frac{\text{FCFF}_t}{(1+WACC)^t} + \frac{\text{TV}_T}{(1+WACC)^T}\Big)$$

We will focus mainly on the Free Cash Flow to the Firm (cash that is available to all stakeholders) and the Free Cash Flow to Equity (cash available to equity holders after all expenses, debt payments, and reinvestment).

$$FCFF_t = EBIT_t(1-\tau) + D\&A_t - CapEx_t - \Delta NWC_t$$

Where:

- $FCFF_t$ = Free Cash Flow to the Firm in period $t$
- $WACC$ = Weighted Average Cost of Capital  
- $TV_T$ = Terminal Value at forecast end
- $\tau$ = tax rate, $D\&A$ = depreciation & amortization

**DCF Challenges:**
- Requires detailed forecasts for 5-10 years
- Terminal value often represents 60-80% of total value  
- Highly sensitive to growth and discount rate assumptions
- Time-intensive and subjective

**Why Multiples Matter:**
- Quick market-based alternative
- Reflects current investor sentiment  
- Useful for cross-sectional comparison
- Can validate DCF assumptions

## The Mathematical Connection: DCF to Multiples (Part 1)

**Step 1: Start with the perpetuity DCF formula**

For a company in steady state with constant growth $g$:

$$EV = \frac{FCFF_1}{WACC - g}$$

**Step 2: Express FCFF in terms of EBITDA**

Starting from our FCFF formula:
$$FCFF = EBIT(1-\tau) + D\&A - CapEx - \Delta NWC$$

Since $EBIT = EBITDA - D\&A$, we can substitute:
$$FCFF = (EBITDA - D\&A)(1-\tau) + D\&A - CapEx - \Delta NWC$$

Expanding and simplifying:
$$FCFF = EBITDA(1-\tau) - D\&A(1-\tau) + D\&A - CapEx - \Delta NWC$$
$$FCFF = EBITDA(1-\tau) + D\&A \cdot \tau - CapEx - \Delta NWC$$

## The Mathematical Connection: DCF to Multiples (Part 2)

**Step 3: Express each component as a ratio of EBITDA**

Let's define these key ratios:
- $\kappa = \frac{CapEx}{EBITDA}$ (CapEx intensity)
- $\delta = \frac{D\&A}{EBITDA}$ (D&A rate)
- $\omega = \frac{\Delta NWC}{EBITDA}$ (Working capital intensity)

**Step 4: Substitute back into the perpetuity formula**

$$FCFF = EBITDA[(1-\tau) + \delta \cdot \tau - \kappa - \omega]$$

Therefore:
$$EV = \frac{EBITDA[(1-\tau) + \delta \cdot \tau - \kappa - \omega]}{WACC - g}$$

**Step 5: Solve for the EV/EBITDA multiple**

$$\frac{EV}{EBITDA} = \frac{(1-\tau) + \delta \cdot \tau - \kappa - \omega}{WACC - g}$$

**Key Insight:** The EV/EBITDA multiple depends on tax rate, CapEx intensity, growth, and risk!

## Practical Example & Sensitivity Analysis

**Given company assumptions:**
- Tax rate: $\tau = 25\%$
- CapEx intensity: $\kappa = 15\%$ of EBITDA
- D&A rate: $\delta = 12\%$ of EBITDA  
- Working capital: $\omega = 2\%$ of EBITDA
- Growth rate: $g = 3\%$, WACC: $10\%$

**Step-by-step calculation:**

$$\frac{EV}{EBITDA} = \frac{(1-0.25) + (0.12 \times 0.25) - 0.15 - 0.02}{0.10 - 0.03}$$

$$= \frac{0.75 + 0.03 - 0.15 - 0.02}{0.07} = \frac{0.61}{0.07} = 8.7x$$

**Sensitivity Analysis:**
- If growth increases to 4%: Multiple = $\frac{0.61}{0.06} = 10.2x$
- If CapEx drops to 10%: Multiple = $\frac{0.66}{0.07} = 9.4x$
- If WACC rises to 12%: Multiple = $\frac{0.61}{0.09} = 6.8x$

**Key Takeaway:** Multiples ARE DCF models in disguise!

## Multiples: Market-Based Valuation

**Common Multiples:**
- **EV/EBITDA**: Enterprise Value to EBITDA
- **P/E**: Price to Earnings
- **P/B**: Price to Book Value
- **P/S**: Price to Sales

- The idea is to believe that the multiple of a firm behaves ***similarly*** to the multiple of a comparable firm. Since even across similar firms, multiples can vary, we compare the multiple of a firm to the average (or median) multiple of its peers.
- The best guess of the multiple of your firm is the average (or median) multiple of its peers.
- Who are these peers?

## Triangulation & LLM Applications

**Why Use Both Methods (when available):**

**DCF provides intrinsic value:**
- Forward-looking and company-specific
- Captures unique growth opportunities
- Independent of market sentiment

**Multiples provide market-implied value:**
- Reflects current market conditions
- Quick and comparable across peers
- Incorporates market expectations

# Let's get started

## Accounting Information of TSLA for Business Valuation

In [1]:
%pip install requests --no-cache-dir

In [ ]:
import requests
from typing import Union, Tuple

def latest_10k_or_10q_text(
    cik: str | int,
    *,
    user_agent: str = "QuickFilingGrabber/2.0 (jfimbett@gmail.com)",
    return_url: bool = False,
) -> Union[str, Tuple[str, str]]:
    """
    Download the newest 10-K or 10-Q for `cik` and return the **raw text**.

    Parameters
    ----------
    cik : str | int
        Company CIK (with or without leading zeros).
    user_agent : str, optional
        REQUIRED by the SEC – include contact info in parentheses.
    return_url : bool, default False
        If True, also return the SEC master-file URL.

    Returns
    -------
    str                    – if return_url is False.
    (str, str) tuple       – if return_url is True (second element is the URL).

    Raises
    ------
    ValueError      – if no recent 10-K/10-Q exists.
    requests.HTTPError – for any HTTP failure.
    """
    cik_raw = str(int(cik))           # strip leading zeros from user input
    cik10   = cik_raw.zfill(10)       # pad back to 10 digits for the API

    # 1) Pull the recent-filings feed -----------------------------------------------
    feed_url = f"https://data.sec.gov/submissions/CIK{cik10}.json"
    hdrs     = {"User-Agent": user_agent, "Accept-Encoding": "gzip, deflate"}
    recent   = requests.get(feed_url, headers=hdrs, timeout=30).json()["filings"]["recent"]

    # 2) Find the first 10-K or 10-Q -------------------------------------------------
    for form, acc in zip(recent["form"], recent["accessionNumber"]):
        if form in {"10-K", "10-Q"}:
            acc_no_dash = acc.replace("-", "")
            txt_url = (
                f"https://www.sec.gov/Archives/edgar/data/{int(cik_raw)}/"
                f"{acc_no_dash}/{acc}.txt"
            )
            raw_txt = requests.get(txt_url, headers=hdrs, timeout=60).text
            break
    else:
        raise ValueError("No recent 10-K or 10-Q found in the feed.")

    return (raw_txt, txt_url) if return_url else raw_txt


# ---------------------------------------------------------------------------
# DEMO: Tesla, Inc. (CIK 0001318605)
# ---------------------------------------------------------------------------

text, url = latest_10k_or_10q_text("0001318605", return_url=True)
print(f"Tesla filing URL: {url}")
print(f"Characters downloaded: {len(text):,}")
print("\nFirst 500 characters:\n")
print(text[:500])


Tesla filing URL: https://www.sec.gov/Archives/edgar/data/1318605/000162828025018911/0001628280-25-018911.txt
Characters downloaded: 7,508,813

First 500 characters:

<SEC-DOCUMENT>0001628280-25-018911.txt : 20250423
<SEC-HEADER>0001628280-25-018911.hdr.sgml : 20250423
<ACCEPTANCE-DATETIME>20250422210210
ACCESSION NUMBER:		0001628280-25-018911
CONFORMED SUBMISSION TYPE:	10-Q
PUBLIC DOCUMENT COUNT:		79
CONFORMED PERIOD OF REPORT:	20250331
FILED AS OF DATE:		20250423
DATE AS OF CHANGE:		20250422

FILER:

	COMPANY DATA:	
		COMPANY CONFORMED NAME:			Tesla, Inc.
		CENTRAL INDEX KEY:			0001318605
		STANDARD INDUSTRIAL CLASSIFICATION:	MOTOR VEHICLES & PASSENGER CAR 


In [ ]:
import math
from typing import List

def split_into_chunks(
    text: str,
    n_chunks: int,
    overlap_frac: float = 0.10,
) -> List[str]:
    """
    Split `text` into `n_chunks`, each overlapping the next by `overlap_frac`.

    Parameters
    ----------
    text : str
        The full raw text you want to slice up.
    n_chunks : int
        Desired number of chunks (≥ 2).
    overlap_frac : float, default 0.10
        Fraction of each chunk that should overlap with its successor.
        Must satisfy 0 ≤ overlap_frac < 1.

    Returns
    -------
    List[str]  – `n_chunks` items; the last chunk may be shorter if the
                 math doesn't land exactly on the final character.
    """
    if not (0 <= overlap_frac < 1):
        raise ValueError("overlap_frac must be in the half-open interval [0, 1).")
    if n_chunks < 2:
        return [text]

    L = len(text)
    # Effective step forward each time (non-overlapping part)
    step = L / n_chunks
    overlap = overlap_frac * step
    stride = step - overlap

    # Round to integers for slicing
    step_i   = math.ceil(step)
    stride_i = max(1, math.ceil(stride))

    chunks = []
    start = 0
    for _ in range(n_chunks):
        end = start + step_i
        chunks.append(text[int(start):int(end)])
        start += stride_i
        if start >= L:
            break

    # If rounding left us short of the target count, pad last chunk(s)
    while len(chunks) < n_chunks:
        chunks.append("")
    return chunks


# ---------------------------------------------------------------------------
# EXAMPLE: grab Tesla’s latest 10-K/10-Q, then slice it into 8 chunks
# ---------------------------------------------------------------------------

chunks = split_into_chunks(text, n_chunks=8, overlap_frac=0.10)
for i, c in enumerate(chunks, 1):
    print(f"\n--- chunk {i}/{len(chunks)} (len={len(c):,}) ---\n")
    print(c[:500])      # preview first 500 chars of each chunk



--- chunk 1/8 (len=938,602) ---

<SEC-DOCUMENT>0001628280-25-018911.txt : 20250423
<SEC-HEADER>0001628280-25-018911.hdr.sgml : 20250423
<ACCEPTANCE-DATETIME>20250422210210
ACCESSION NUMBER:		0001628280-25-018911
CONFORMED SUBMISSION TYPE:	10-Q
PUBLIC DOCUMENT COUNT:		79
CONFORMED PERIOD OF REPORT:	20250331
FILED AS OF DATE:		20250423
DATE AS OF CHANGE:		20250422

FILER:

	COMPANY DATA:	
		COMPANY CONFORMED NAME:			Tesla, Inc.
		CENTRAL INDEX KEY:			0001318605
		STANDARD INDUSTRIAL CLASSIFICATION:	MOTOR VEHICLES & PASSENGER CAR 

--- chunk 2/8 (len=938,602) ---

74pt"><span style="color:#000000;font-family:'Times New Roman',sans-serif;font-size:10pt;font-weight:400;line-height:120%">The following is a summary of our debt and finance leases as of December&#160;31, 2024 (in millions):</span></div><div style="margin-top:10pt"><table style="border-collapse:collapse;display:inline-table;margin-bottom:5pt;vertical-align:text-bottom;width:100.000%"><tr><td style="width:1.0%"/><td style="width

In [ ]:
!pip install transformers tqdm
from transformers import pipeline
from typing import List, Dict, Tuple
from tqdm.auto import tqdm


def extract_financial_values(
    chunks: List[str],
    *,
    # variable → question phrasing
    questions: Dict[str, str] | None = None,
    model_name: str = "deepset/roberta-base-squad2",
    device: int | str | None = None,       # 0, 1, … for GPUs, or "cpu"
    answer_threshold: float = 0.20,        # ignore very low-confidence spans
    progress_desc: str = "Answering questions on chunks",
) -> Dict[str, Dict[str, Tuple]]:
    """
    Ask each `questions[var]` about every chunk and keep the highest-
    confidence answer for every variable.

    Returns
    -------
    {
        "EBIT": {
            "answer": "1,234 million",
            "score": 0.87,
            "chunk_idx": 3
        },
        ...
    }
    """
    if questions is None:
        questions = {
            "EBIT": "What is the company's EBIT in 2025Q1?",
            "Taxes": "What is the company's income-tax expense in 2025Q1?",
            "Depreciation and Amortization": "What is depreciation and amortization in 2025Q1?",
            "Capex": "What were capital expenditures (capex) in in 2025Q1?",
            "Working Capital": "What is the change in working capital in the last quarter?",
        }

    qa = pipeline("question-answering", model=model_name, device=device)

    # initialise best-answer store
    best: Dict[str, Dict[str, Tuple]] = {
        var: {"answer": None, "score": -1.0, "chunk_idx": None}
        for var in questions
    }

    for idx, chunk in enumerate(tqdm(chunks, desc=progress_desc, unit="chunk")):
        for var, question in questions.items():
            res = qa(question=question, context=chunk)
            if res["score"] >= answer_threshold and res["score"] > best[var]["score"]:
                best[var] = {
                    "answer": res["answer"].strip(),
                    "score": res["score"],
                    "chunk_idx": idx,
                }

    return best


# ─── Example usage ──────────────────────────────────────────────────────────────
chunks = split_into_chunks(text, n_chunks=8, overlap_frac=0.10)
answers = extract_financial_values(chunks, device=0)

for var, info in answers.items():
    print(f"{var:30s}  {info['answer']}  (score={info['score']:.2f}, "
          f"from chunk {info['chunk_idx']})")


DEPRECATION: pytorch-lightning 1.6.5 has a non-standard dependency specifier torch>=1.8.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063


ModuleNotFoundError: No module named 'transformers'

## Using OpenAI with Collab Secrets

In [ ]:
import os, json, backoff
from typing import List, Dict, Tuple, Any
from tqdm.auto import tqdm
from openai import (                         # ≥1.0 SDK
    OpenAI,
    RateLimitError,
    APITimeoutError,
    APIConnectionError,
    APIStatusError,
)
from google.colab import userdata
# --------------------------------------------------------------------------- #
#  Initialise client & retry wrapper                                          #
# --------------------------------------------------------------------------- #
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
retry_errors = (RateLimitError, APITimeoutError,
                APIConnectionError, APIStatusError)

@backoff.on_exception(backoff.expo, retry_errors, max_time=60)
def _ask_openai(messages, model="gpt-4o-mini") -> str:
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
        max_tokens=64,
    )
    return resp.choices[0].message.content.strip()

# --------------------------------------------------------------------------- #
#  Main extractor – returns *all* answers                                      #
# --------------------------------------------------------------------------- #
def extract_financial_values_openai(
    chunks: List[str],
    *,
    questions: Dict[str, str] | None = None,
    model_name: str = "gpt-4o-mini",   # 128 k-token context
    answer_threshold: float = 0.30,
    progress_desc: str = "Querying OpenAI on chunks",
) -> Dict[str, List[Dict[str, Any]]]:
    """
    Ask every question about every chunk.

    Returns
    -------
    {
        "EBIT": [
            {"answer": "1,234 million", "score": 0.87, "chunk_idx": 2},
            {"answer": "1.21 bn",       "score": 0.56, "chunk_idx": 5},
            ...
        ],
        "Taxes": [ ... ],
        ...
    }
    (Each list is sorted by score descending.)
    """
    if questions is None:
        questions = {
            "EBIT": "What is the company's EBIT in 2025 Q1?",
            "Taxes": "What is the income-tax expense in 2025 Q1?",
            "Depreciation and Amortization": "What is depreciation and amortization in 2025 Q1?",
            "Capex": "What were capital expenditures (capex) in 2025 Q1?",
            "Working Capital": "What is the change in working capital in the last quarter?",
        }

    # accumulate every qualifying answer
    results: Dict[str, List[Dict[str, Any]]] = {v: [] for v in questions}

    system_msg = {
        "role": "system",
        "content": (
            "You are a meticulous financial-statement analyst. "
            "Given an excerpt from a 10-K or 10-Q, answer the question. Always state units. "
            'If the excerpt lacks the information, answer "N/A" with confidence 0. '
            'Respond ONLY with JSON like {"answer": "...", "confidence": 0-1}.'
        )
    }

    for idx, chunk in enumerate(tqdm(chunks, desc=progress_desc, unit="chunk")):
        for var, question in questions.items():
            user_msg = {
                "role": "user",
                "content": f"Context:\n\"\"\"\n{chunk}\n\"\"\"\n\nQuestion: {question}",
            }
            try:
                data = json.loads(_ask_openai([system_msg, user_msg], model=model_name))
            except (json.JSONDecodeError, KeyError):
                data = {"answer": "N/A", "confidence": 0.0}

            score = float(data.get("confidence", 0))
            if score >= answer_threshold:
                results[var].append(
                    {"answer": str(data.get("answer", "N/A")).strip(),
                     "score": score,
                     "chunk_idx": idx}
                )

    # sort each list by descending confidence
    for var in results:
        results[var].sort(key=lambda d: d["score"], reverse=True)

    return results


## Deal with large context sizes

- See how openai responds to more tokens than the context length
- Adapt accordingly

In [ ]:

# !pip install tiktoken   # run once if not installed

import tiktoken

def count_tokens(text: str, model: str = "gpt-4o-mini") -> int:
    """
    Return the number of tokens `text` would occupy for the given OpenAI model.
    Falls back to the GPT-4/3.5 tokenizer ('cl100k_base') if the model name
    isn't recognised by tiktoken.
    """
    try:
        enc = tiktoken.encoding_for_model(model)
    except KeyError:
        enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(text))

chunks = split_into_chunks(text, n_chunks=50, overlap_frac=0.05)

lenghts = [count_tokens(chunk) for chunk in chunks]

# maximum number of tokens
print(f"Maximum number of tokens: {max(lenghts)}")


### Try only a subset of chunks in class

In [ ]:
N = min(len(chunks), 2)
chunks_subset = chunks[:N]
answers = extract_financial_values_openai(chunks_subset)

for var, candidates in answers.items():
    print(f"\n{var}:")
    for c in candidates:
        print(f"  {c['answer']:>25}  (score={c['score']:.2f}, from chunk {c['chunk_idx']})")


# RAG

- **Retrieval-Augmented Generation (RAG)** is an LLM pattern that first *retrieves* relevant documents (or text chunks) from an external knowledge store, then passes those snippets into the model’s prompt so it can *generate* an answer grounded in that evidence.  
- By pulling context on-demand instead of relying purely on parameters, RAG **reduces hallucinations** and keeps answers up-to-date without retraining or fine-tuning the model itself.  
- The retrieval step typically uses **vector similarity search** (embeddings → FAISS, Pinecone, pgvector, etc.), optionally combined with keyword or hybrid ranking to pick the top-k most relevant chunks.  
- The generation step feeds the retrieved text into a chat/completions model with an instruction like “Using only the provided context, answer…”, producing a coherent, cited response.  
- Because the knowledge lives outside the model, RAG solutions are **easier to maintain** (swap in new documents) and **cheaper** than large-scale fine-tuning, while still letting you add citations, filters, and domain-specific reasoning on the fly.


In [ ]:
from openai import OpenAI          # new-style client class
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

assistant = client.beta.assistants.create(
  name="Financial Analyst Assistant",
  instructions="You are an expert financial analyst. Use you knowledge base to answer questions about audited financial statements.",
  model="gpt-4o-mini",
  tools=[{"type": "file_search"}],
)

In [ ]:
import openai
# store all text in a .txt file
with open("tesla_10k.txt", "w") as f:
    f.write(text)

# Create a vector store caled "Financial Statements"
vector_store = client.vector_stores.create(name="Financial Statements")

# Ready the files for upload to OpenAI
file_paths = ["tesla_10k.txt"]
file_streams = [open(path, "rb") for path in file_paths]

# Use the upload and poll SDK helper to upload the files, add them to the vector store,
# and poll the status of the file batch for completion.
file_batch = client.vector_stores.file_batches.upload_and_poll(
  vector_store_id=vector_store.id, files=file_streams
)

# You can print the status and the file counts of the batch to see the result of this operation.
print(file_batch.status)
print(file_batch.file_counts)

assistant = client.beta.assistants.update(
  assistant_id=assistant.id,
  tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}},
)

# Upload the user provided file to OpenAI
message_file = client.files.create(
  file=open("tesla_10k.txt", "rb"), purpose="assistants"
)




## Ready to stream

In [ ]:
from typing_extensions import override
from openai import AssistantEventHandler, OpenAI


client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# Create a thread and attach the file to the message
thread = client.beta.threads.create(
  messages=[
    {
      "role": "user",
      "content": "What was the CAPEX of TSLA in 2024?",
      # Attach the new file to the message.
      "attachments": [
        { "file_id": message_file.id, "tools": [{"type": "file_search"}] }
      ],
    }
  ]
)

# The thread now has a vector store with that file in its tool resources.
print(thread.tool_resources.file_search)

class EventHandler(AssistantEventHandler):
    @override
    def on_text_created(self, text) -> None:
        print(f"\nassistant > ", end="", flush=True)

    @override
    def on_tool_call_created(self, tool_call):
        print(f"\nassistant > {tool_call.type}\n", flush=True)

    @override
    def on_message_done(self, message) -> None:
        # print a citation to the file searched
        message_content = message.content[0].text
        annotations = message_content.annotations
        citations = []
        for index, annotation in enumerate(annotations):
            message_content.value = message_content.value.replace(
                annotation.text, f"[{index}]"
            )
            if file_citation := getattr(annotation, "file_citation", None):
                cited_file = client.files.retrieve(file_citation.file_id)
                citations.append(f"[{index}] {cited_file.filename}")

        print(message_content.value)
        print("\n".join(citations))

# Then, we use the stream SDK helper
# with the EventHandler class to create the Run
# and stream the response.

name = "Juan Imbet"

with client.beta.threads.runs.stream(
    thread_id=thread.id,
    assistant_id=assistant.id,
    instructions=f"Please address the user as {name}. The user has a premium account.",
    event_handler=EventHandler(),
) as stream:
    stream.until_done()

## Document embeddings

In [ ]:
response = client.embeddings.create(
    input="This is some text",
    model="text-embedding-3-small"
)

print(response.data[0].embedding)

In [ ]:
import os
import openai
from pathlib import Path
import tiktoken   # optional: helps count/trim tokens


MODEL = "text-embedding-3-large"  # max 8000 tokens

chunks = split_into_chunks(text, n_chunks=700, overlap_frac=0.05)

lenghts = [count_tokens(chunk) for chunk in chunks]

# maximum number of tokens
print(f"Maximum number of tokens: {max(lenghts)}")



In [ ]:
# Embed each chunk
embeddings = []
for chunk in tqdm(chunks):
    resp = client.embeddings.create(model=MODEL, input=chunk)
    embeddings.append(resp.data[0].embedding)

# You now have a list of vectors, one per chunk
print(f"{len(embeddings)=}, dimension={len(embeddings[0])}")



In [ ]:
import numpy as np

# embeddings is your list[ list[float] ]
embeddings_np = np.array(embeddings, dtype=np.float32)   # shape: (num_chunks, dim)

# 1. Average across rows (chunks)
doc_vec = embeddings_np.mean(axis=0)                     # shape: (dim,)

# 2. Re-normalize to unit length (cosine-sim friendly)
doc_vec /= np.linalg.norm(doc_vec)

print(f"Document vector dimension: {doc_vec.shape[0]}, norm: {np.linalg.norm(doc_vec):.3f}")
